# Water Potability: Classical vs Quantum Kernel Pipeline
### Leakage-corrected protocol

## What changed and why

**1. Imputation moved inside the split.** It was previously fitted on all 10,000
rows in `quality_control`, so test-set column medians influenced the training
representation. It now lives in `fit_preprocessing`, fitted on the training
partition and applied to validation and test. Deduplication and
physical-plausibility bounds stay before the split: those use fixed domain
constants, not data-estimated statistics, so they leak nothing.

**2. EDA moved to the training partition.** Mutual information, correlations and
moments are all estimated quantities. Computing them on the full table lets
test-set structure inform reported findings even when no model consumes them.
EDA now runs on the 6,000-row training partition of the first seed, and says so
in its header.

**3. Bottleneck reference model selected on validation.** It was previously
chosen by test balanced accuracy, which made every downstream statement
conditional on the test labels. It now uses the validation scores returned by
the classical stage.

**4. Imbalance strategy selected on validation.** Compares none, class weighting,
SMOTE, random oversampling and random undersampling on the validation partition.
Resampling is applied to the training partition only. On this balanced table the
procedure correctly selects `none` — reporting that selection is more
informative than asserting resampling was unnecessary.

**5. Quantum stage verified.** PCA, qubit selection and the `[0, π]` scaler were
already fitted on training data and scored on validation, which the audit
confirmed. Two changes: the search loop no longer transforms the test partition
at all, and the stage prints a fit-order verification before running.

## Leakage audit

The pipeline prints this table at the first seed:

| Fitted object | Partition |
|---|---|
| SimpleImputer, StandardScaler | train |
| Mutual information | train |
| PCA / qubit selection | train |
| MinMaxScaler → [0, π] | train |
| Classical hyperparameters | validation |
| Imbalance strategy | validation |
| Bottleneck reference model | validation |
| QSVM hyperparameters | validation |
| Reported metrics | test (evaluated once) |

## Settings

```python
CONFIG = {
    "data": None,          # None -> kagglehub; or "/path/to/file.csv"
    "seeds": [42, 7, 2024],
    "max-train": 3000,     # quantum kernel is O(N^2)
    "outdir": "results",
    "quick": False,
}
```

`pip install imbalanced-learn` to enable the full imbalance comparison; without
it the selector falls back to comparing none against class weighting.

In [1]:
"""
================================================================================
WATER POTABILITY: CLASSICAL vs QUANTUM KERNEL PIPELINE
================================================================================

Implements the study flowchart end to end:

    DATASET -> EDA -> DATA QUALITY CONTROL -> STRATIFIED TRAIN/VAL/TEST
    -> COMMON PREPROCESSING
        |-> CLASSICAL PIPELINE (LogReg, RBF-SVM, RandomForest, XGBoost)
        |       -> CLASSICAL PERFORMANCE -> BOTTLENECK ANALYSIS
        |          (minority errors / feature overlap / generalization)
        |       -> QUANTUM HYPOTHESIS
        \\-> QUANTUM PIPELINE
                PCA representation -> automatic component/qubit selection
                -> quantum normalization [0, pi] -> ZZFeatureMap
                -> fidelity quantum kernel (block-wise) -> QSVM
                -> validation optimization -> frozen QSVM
    -> SAME UNTOUCHED TEST SET -> DIRECT COMPARISON (9 metrics)
    -> STATISTICAL ANALYSIS -> MULTI-SEED ROBUSTNESS -> ABLATION STUDIES

Usage
-----
    pip install kagglehub numpy pandas scikit-learn scipy matplotlib seaborn xgboost
    python pipeline.py
    python pipeline.py --data /path/to/file.csv --seeds 42 7 2024
    python pipeline.py --quick          # single seed, reduced grids

Notebook: edit CONFIG below, then run the cell.
"""
from __future__ import annotations

import argparse
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, matthews_corrcoef, precision_recall_curve,
    precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

sns.set_theme(style="whitegrid", context="paper")
TARGET = "Potability"
C0, C1, CQ = "#5B6C8F", "#3F8F7A", "#B4694A"
METRICS = ["accuracy", "balanced_accuracy", "precision", "recall", "f1",
           "roc_auc", "pr_auc", "mcc", "specificity"]

CONFIG = {
    "data": None,            # None -> kagglehub; or "/path/to/file.csv"
    "seeds": [42, 7, 2024],
    "max-train": 3000,       # quantum kernel is O(N^2)
    "outdir": "results",
    "quick": False,
}


def _cli_args():
    """CLI args, or CONFIG when running inside a notebook kernel."""
    import sys
    in_nb = False
    try:
        from IPython import get_ipython
        ip = get_ipython()
        in_nb = ip is not None and ip.__class__.__name__ == "ZMQInteractiveShell"
    except Exception:
        pass
    if not in_nb:
        in_nb = any(a.endswith(".json") or "ipykernel" in a for a in sys.argv[1:])
    if not in_nb:
        return None
    argv = []
    for k, v in CONFIG.items():
        if v is None or v is False:
            continue
        if v is True:
            argv.append(f"--{k}"); continue
        argv.append(f"--{k}")
        argv += [str(x) for x in (v if isinstance(v, (list, tuple)) else [v])]
    return argv


def banner(text):
    print(f"\n{'='*78}\n{text}\n{'='*78}", flush=True)


# ═══════════════════════════════════════════════════════════════════════════ #
# QUANTUM CORE
# ═══════════════════════════════════════════════════════════════════════════ #
def entangling_pairs(n, ent="linear"):
    if ent == "linear":
        return [(i, i + 1) for i in range(n - 1)]
    if ent == "circular":
        p = [(i, i + 1) for i in range(n - 1)]
        return p + [(n - 1, 0)] if n > 2 else p
    if ent == "full":
        return [(i, j) for i in range(n) for j in range(i + 1, n)]
    if ent == "none":
        return []
    raise ValueError(ent)


def _bits(n):
    idx = np.arange(2 ** n)
    return ((idx[:, None] >> np.arange(n)[None, :]) & 1).astype(np.int8)


def _hadamard(psi, n):
    out, h, m = psi, 1, psi.shape[0]
    for _ in range(n):
        out = out.reshape(m, -1, 2 * h)
        a, b = out[:, :, :h].copy(), out[:, :, h:].copy()
        out[:, :, :h], out[:, :, h:] = a + b, a - b
        out = out.reshape(m, -1)
        h *= 2
    return out / np.sqrt(2 ** n)


def zz_statevectors(X, reps=2, ent="linear", alpha=2.0):
    """ZZFeatureMap encoding. The map is a Hadamard layer plus a diagonal phase
    operator, so states have a closed form and no circuit simulation is needed."""
    X = np.asarray(X, float)
    ns, nq = X.shape
    bits = _bits(nq)
    theta = alpha * (X @ bits.T.astype(float))
    pairs = entangling_pairs(nq, ent)
    if pairs:
        pi_ = np.array([p[0] for p in pairs]); pj_ = np.array([p[1] for p in pairs])
        parity = (bits[:, pi_] ^ bits[:, pj_]).astype(float)
        theta = theta + (alpha * (np.pi - X[:, pi_]) * (np.pi - X[:, pj_])) @ parity.T
    phase = np.exp(1j * theta)
    psi = np.zeros(2 ** nq, complex); psi[0] = 1.0
    psi = np.broadcast_to(psi, (ns, 2 ** nq)).copy()
    for _ in range(reps):
        psi = _hadamard(psi, nq); psi *= phase
    return psi


def fidelity_kernel(Xa, Xb=None, reps=2, ent="linear", alpha=2.0, block=1024):
    """K(x,y) = |<phi(x)|phi(y)>|^2, computed block-wise to bound peak memory."""
    sym = Xb is None
    Pa = zz_statevectors(Xa, reps, ent, alpha)
    Pb = Pa if sym else zz_statevectors(Xb, reps, ent, alpha)
    na, nb = len(Pa), len(Pb)
    K = np.empty((na, nb), float)
    for i0 in range(0, na, block):
        i1 = min(i0 + block, na)
        for j0 in range(0, nb, block):
            j1 = min(j0 + block, nb)
            K[i0:i1, j0:j1] = np.abs(Pa[i0:i1] @ Pb[j0:j1].conj().T) ** 2
    if sym:
        K = 0.5 * (K + K.T); np.fill_diagonal(K, 1.0)
    return K


def kernel_diagnostics(K, y):
    n = len(K); off = ~np.eye(n, dtype=bool)
    same = (y[:, None] == y[None, :]) & off
    eig = np.clip(np.linalg.eigvalsh(K), 0, None)
    p = eig / max(eig.sum(), 1e-12); p = p[p > 0]
    return {"offdiag_mean": float(K[off].mean()),
            "within_class": float(K[same].mean()),
            "between_class": float(K[~same & off].mean()),
            "effective_rank": float(np.exp(-(p * np.log(p)).sum()))}


def kernel_target_alignment(K, y):
    yy = np.where(y == 1, 1.0, -1.0); T = np.outer(yy, yy); n = len(K)
    H = np.eye(n) - np.ones((n, n)) / n; Kc = H @ K @ H
    return float((Kc * T).sum() / max(np.sqrt((Kc * Kc).sum() * (T * T).sum()), 1e-12))


# ═══════════════════════════════════════════════════════════════════════════ #
# STAGE 1-2: DATASET + EDA
# ═══════════════════════════════════════════════════════════════════════════ #
def load_dataset(path=None):
    banner("STAGE 1 | DATASET")
    if path:
        p = Path(path); csv = p if p.is_file() else sorted(p.rglob("*.csv"))[0]
    else:
        import kagglehub
        d = kagglehub.dataset_download("tamilamudan/water-quality-potability")
        print("Path to dataset files:", d)
        csv = sorted(Path(d).rglob("*.csv"))[0]
    df = pd.read_csv(csv)
    print(f"Loaded {csv.name}: {df.shape[0]} rows x {df.shape[1]} columns")
    return df


def run_eda(df, feats, out, partition_label='training partition'):
    banner(f"STAGE 2 | EXPLORATORY DATA ANALYSIS ({partition_label})")
    print(f"EDA computed on {len(df)} rows. Every estimated quantity below\n"
          "(MI, correlations, moments) is fitted on this partition only.")
    vc = df[TARGET].value_counts().sort_index()
    print(f"Class balance: {vc.to_dict()}  ratio {vc.max()/vc.min():.2f}:1")
    print(f"Majority baseline accuracy: {vc.max()/len(df):.4f}")
    print(f"Duplicates: {int(df.duplicated().sum())}   "
          f"Missing cells: {int(df.isna().sum().sum())}")

    rows = []
    for c in feats:
        s = df[c].dropna()
        q1, q3 = s.quantile(.25), s.quantile(.75); iqr = q3 - q1
        outl = (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)
        rows.append({"feature": c, "mean": s.mean(), "sd": s.std(),
                     "median": s.median(), "skew": stats.skew(s),
                     "kurtosis": stats.kurtosis(s),
                     "shapiro_p": stats.shapiro(s.sample(min(5000, len(s)),
                                                         random_state=42)).pvalue,
                     "n_outliers": int(outl.sum()),
                     "pct_outliers": 100 * outl.mean()})
    desc = pd.DataFrame(rows)
    desc.to_csv(out / "tables" / "descriptives.csv", index=False)
    print("\nDescriptives:")
    print(desc[["feature", "mean", "sd", "skew", "kurtosis",
                "pct_outliers"]].round(3).to_string(index=False))

    grp = []
    for c in feats:
        a = df.loc[df[TARGET] == 0, c].dropna(); b = df.loc[df[TARGET] == 1, c].dropna()
        u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
        grp.append({"feature": c, "median_0": a.median(), "median_1": b.median(),
                    "p": p, "rank_biserial": 1 - 2 * u / (len(a) * len(b)),
                    "cohens_d": (b.mean() - a.mean()) / np.sqrt((a.var() + b.var()) / 2)})
    g = pd.DataFrame(grp)
    m = len(g); order = np.argsort(g["p"].values); adj = np.empty(m); run = 0.0
    for r, i in enumerate(order):
        run = max(run, (m - r) * g["p"].values[i]); adj[i] = min(run, 1.0)
    g["p_holm"] = adj; g["significant"] = adj < 0.05
    g = g.sort_values("p")
    g.to_csv(out / "tables" / "group_tests.csv", index=False)
    print("\nFeature vs target (Mann-Whitney, Holm-corrected):")
    print(g[["feature", "median_0", "median_1", "p_holm", "cohens_d",
             "significant"]].round(4).to_string(index=False))
    print(f"  {int(g['significant'].sum())}/{len(g)} significant after correction")

    corr = df[feats + [TARGET]].corr(method="spearman")
    corr.to_csv(out / "tables" / "correlation.csv")

    # Mutual information: dependence that is not necessarily monotone, so it
    # detects structure the Spearman coefficient and the mean-shift tests miss.
    mi_tab, mi_ff = mutual_information(df, feats, out=out)

    fig_dataset_overview(df, feats, desc, out)
    fig_distributions(df, feats, out)
    fig_outliers(df, feats, desc, out)
    fig_correlation_mi(corr, mi_tab, mi_ff, feats, out)
    fig_mutual_information(mi_tab, g, out)
    return desc, g, corr, mi_tab


# ═══════════════════════════════════════════════════════════════════════════ #
# STAGE 3: DATA QUALITY CONTROL
# ═══════════════════════════════════════════════════════════════════════════ #
def quality_control(df, feats, out):
    """Non-learned quality control only.

    Deduplication and physical-plausibility bounds use fixed domain constants,
    not statistics estimated from the data, so applying them before the split
    leaks nothing. Imputation DOES estimate from data and is therefore deferred
    to fit_preprocessing, which sees the training partition only.
    """
    banner("STAGE 3 | DATA QUALITY CONTROL (non-learned)")
    n0 = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Duplicate removal: {n0} -> {len(df)} rows")

    bounds = {"ph": (0, 14), "Hardness": (0, 1000), "Solids": (0, 100000),
              "Chloramines": (0, 30), "Sulfate": (0, 1000), "Conductivity": (0, 2000),
              "Organic_carbon": (0, 100), "Trihalomethanes": (0, 300),
              "Turbidity": (0, 20)}
    n_bad = 0
    for c, (lo, hi) in bounds.items():
        if c in df.columns:
            bad = ((df[c] < lo) | (df[c] > hi)) & df[c].notna()
            if bad.any():
                print(f"  {int(bad.sum())} implausible values in {c} -> NaN")
                df.loc[bad, c] = np.nan
                n_bad += int(bad.sum())
    print(f"Implausible values recoded: {n_bad}")
    n_miss = int(df[feats].isna().sum().sum())
    print(f"Missing cells remaining: {n_miss} "
          f"(imputed later, fitted on training partition only)")

    json.dump({"rows_after_qc": int(len(df)), "duplicates_removed": n0 - len(df),
               "implausible_recoded": n_bad, "missing_after_qc": n_miss,
               "imputation": "deferred to training partition"},
              open(out / "tables" / "quality_control.json", "w"), indent=2)
    return df


# ═══════════════════════════════════════════════════════════════════════════ #
# STAGE 4-5: SPLIT + COMMON PREPROCESSING
# ═══════════════════════════════════════════════════════════════════════════ #
def stratified_split(X, y, seed):
    Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=.4, stratify=y,
                                            random_state=seed)
    Xva, Xte, yva, yte = train_test_split(Xtmp, ytmp, test_size=.5, stratify=ytmp,
                                          random_state=seed)
    return Xtr, Xva, Xte, ytr, yva, yte


def fit_preprocessing(Xtr, Xva, Xte):
    """Fit imputer and scaler on the training partition, apply to all three.

    Both estimate parameters from data (column medians, means and standard
    deviations). Fitting either on the full table before splitting lets test-set
    statistics influence the training representation, which inflates reported
    performance by an amount that is small, systematic, and impossible to detect
    after the fact.
    """
    from sklearn.impute import SimpleImputer

    imp = SimpleImputer(strategy="median").fit(Xtr)
    Itr, Iva, Ite = imp.transform(Xtr), imp.transform(Xva), imp.transform(Xte)
    sc = StandardScaler().fit(Itr)
    return sc.transform(Itr), sc.transform(Iva), sc.transform(Ite), (imp, sc)


def leakage_audit(n_tr, n_va, n_te):
    """State explicitly what each fitted object saw."""
    print("\nLeakage audit — every fitted object and its fitting partition:")
    for name, part in [
        ("SimpleImputer (median)", "train"),
        ("StandardScaler", "train"),
        ("Mutual information", "train"),
        ("PCA (component/qubit selection)", "train"),
        ("MinMaxScaler -> [0, pi]", "train"),
        ("Classical hyperparameters", "validation"),
        ("Imbalance strategy", "validation"),
        ("Bottleneck reference model", "validation"),
        ("QSVM hyperparameters", "validation"),
        ("Reported metrics", "test (evaluated once)"),
    ]:
        print(f"  {name:36s} fitted on: {part}")
    print(f"  partition sizes: train={n_tr}  val={n_va}  test={n_te}")


# ═══════════════════════════════════════════════════════════════════════════ #
# STAGE 6-7: CLASSICAL PIPELINE + PERFORMANCE
# ═══════════════════════════════════════════════════════════════════════════ #
def evaluate(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_true, y_score),
            "pr_auc": average_precision_score(y_true, y_score),
            "mcc": matthews_corrcoef(y_true, y_pred),
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}


def _score(m, X):
    return m.predict_proba(X)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(X)


def select_imbalance_strategy(Atr, Ava, ytr, yva, seed, verbose=True):
    """Choose a class-imbalance strategy on the validation partition.

    Included for completeness of the protocol. On a balanced table the correct
    choice is no resampling, and the procedure should return exactly that —
    reporting the selection is more informative than asserting it was
    unnecessary. Resampling is applied to the training partition only, never to
    validation or test, since resampling before the split is the standard route
    to inflated results.
    """
    counts = np.bincount(ytr)
    ratio = counts.max() / max(counts.min(), 1)
    strategies = {"none": None, "class_weight": "balanced"}
    try:
        from imblearn.over_sampling import SMOTE, RandomOverSampler
        from imblearn.under_sampling import RandomUnderSampler
        strategies.update({"smote": SMOTE(random_state=seed),
                           "random_over": RandomOverSampler(random_state=seed),
                           "random_under": RandomUnderSampler(random_state=seed)})
    except ImportError:
        if verbose:
            print("  imbalanced-learn not installed; comparing none vs class_weight")

    rows = []
    for name, obj in strategies.items():
        Xs, ys, cw = Atr, ytr, None
        if name == "class_weight":
            cw = "balanced"
        elif obj is not None:
            Xs, ys = obj.fit_resample(Atr, ytr)
        m = LogisticRegression(max_iter=5000, class_weight=cw,
                               random_state=seed).fit(Xs, ys)
        rows.append({"strategy": name, "n_train": len(ys),
                     "val_balanced_accuracy": balanced_accuracy_score(yva, m.predict(Ava))})
    tab = pd.DataFrame(rows).sort_values("val_balanced_accuracy", ascending=False)
    best = tab.iloc[0]["strategy"]
    if verbose:
        print(f"  imbalance ratio in training partition: {ratio:.2f}:1")
        print(tab.round(4).to_string(index=False))
        print(f"  selected on validation: {best}")
    return best, tab


def classical_pipeline(Atr, Ava, Ate, ytr, yva, yte, seed, quick=False,
                       imbalance='none'):
    banner("STAGE 6-7 | CLASSICAL PIPELINE")
    zoo = {
        "LogisticRegression": (LogisticRegression(max_iter=5000, random_state=seed),
                               {"C": [0.01, 0.1, 1, 10, 100]}),
        "RBF-SVM": (SVC(probability=True, random_state=seed),
                    {"C": [1, 10, 100], "gamma": ["scale", 0.1, 0.5]}),
        "RandomForest": (RandomForestClassifier(random_state=seed, n_jobs=-1),
                         {"n_estimators": [500], "min_samples_leaf": [1, 3, 8]}),
    }
    if HAS_XGB:
        zoo["XGBoost"] = (XGBClassifier(tree_method="hist", eval_metric="logloss",
                                        random_state=seed, n_jobs=-1, verbosity=0),
                          {"n_estimators": [600], "learning_rate": [0.05],
                           "max_depth": [3, 5, 7], "subsample": [0.8]})
    from itertools import product

    # Resampling touches the training partition only.
    Xs, ys, cw = Atr, ytr, None
    if imbalance == "class_weight":
        cw = "balanced"
    elif imbalance != "none":
        from imblearn.over_sampling import SMOTE, RandomOverSampler
        from imblearn.under_sampling import RandomUnderSampler
        obj = {"smote": SMOTE(random_state=seed),
               "random_over": RandomOverSampler(random_state=seed),
               "random_under": RandomUnderSampler(random_state=seed)}[imbalance]
        Xs, ys = obj.fit_resample(Atr, ytr)
    print(f"  imbalance strategy: {imbalance} (training partition: "
          f"{len(ytr)} -> {len(ys)} rows)")

    rows, frozen, val_scores = [], {}, {}
    for name, (est, grid) in zoo.items():
        keys = list(grid); best, bs, bp = None, -np.inf, None
        for vals in product(*[grid[k] for k in keys]):
            p = dict(zip(keys, vals))
            kw = {**est.get_params(), **p}
            if cw is not None and "class_weight" in kw:
                kw["class_weight"] = cw
            m = est.__class__(**kw).fit(Xs, ys)
            s = balanced_accuracy_score(yva, m.predict(Ava))
            if s > bs:
                best, bs, bp = m, s, p
        r = evaluate(yte, best.predict(Ate), _score(best, Ate))
        rows.append({"model": name, "val_bacc": bs, "params": json.dumps(bp, default=str), **r})
        frozen[name] = best
        val_scores[name] = bs
        print(f"  {name:20s} val={bs:.4f}  test_acc={r['accuracy']:.4f} "
              f"auc={r['roc_auc']:.4f} f1={r['f1']:.4f} mcc={r['mcc']:.4f}", flush=True)
    return pd.DataFrame(rows), frozen, val_scores


# ═══════════════════════════════════════════════════════════════════════════ #
# STAGE 8: BOTTLENECK ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════ #
def bottleneck_analysis(frozen, val_scores, Atr, Ate, ytr, yte, feats, out):
    """Where does the classical arm fail? Three axes: per-class errors, local
    class overlap, and the train/test generalization gap. This is what motivates
    the quantum hypothesis rather than assuming it."""
    banner("STAGE 8 | BOTTLENECK ANALYSIS")
    # Reference model chosen on validation. Picking it by test performance
    # would make every downstream statement conditional on the test labels.
    best_name = max(val_scores, key=val_scores.get)
    m = frozen[best_name]
    pred, score = m.predict(Ate), _score(m, Ate)
    err = pred != yte
    print(f"Reference model (selected on validation): {best_name} "
          f"(val bal-acc {val_scores[best_name]:.4f})")

    per_class = []
    for c in (0, 1):
        sel = yte == c
        per_class.append({"class": c, "n": int(sel.sum()),
                          "error_rate": float(err[sel].mean())})
    print("\n[1] Per-class error")
    for r in per_class:
        print(f"    class {r['class']}: n={r['n']}  error={r['error_rate']:.4f}")
    minority = int(np.bincount(yte).argmin())
    print(f"    minority class ({minority}) error premium: "
          f"{per_class[minority]['error_rate'] - per_class[1-minority]['error_rate']:+.4f}")

    nn = NearestNeighbors(n_neighbors=26).fit(Atr)
    _, idx = nn.kneighbors(Ate)
    agree = (ytr[idx[:, 1:]] == yte[:, None]).mean(axis=1)
    overlap_frac = float((agree < 0.5).mean())
    print("\n[2] Feature-space class overlap")
    print(f"    mean local label agreement (k=25): {agree.mean():.4f}")
    print(f"    test points in majority-disagreeing neighbourhoods: {overlap_frac:.4f}")
    print(f"    error rate inside those neighbourhoods: {err[agree < 0.5].mean():.4f}")
    print(f"    error rate outside: {err[agree >= 0.5].mean():.4f}")

    gap = float(balanced_accuracy_score(ytr, m.predict(Atr))
                - balanced_accuracy_score(yte, m.predict(Ate)))
    print("\n[3] Generalization / stability")
    print(f"    train bal-acc {balanced_accuracy_score(ytr, m.predict(Atr)):.4f}  "
          f"test bal-acc {balanced_accuracy_score(yte, m.predict(Ate)):.4f}  "
          f"gap {gap:+.4f}")

    s = (score - np.median(score)) / (score.std() + 1e-12)
    band = pd.qcut(np.abs(s), 4, labels=["nearest", "near", "far", "farthest"])
    by_band = pd.DataFrame({"band": band, "err": err}).groupby("band", observed=True)["err"].mean()
    print("\n    error rate by distance from decision boundary:")
    print("   ", by_band.round(4).to_dict())

    res = {"reference_model": best_name, "per_class_error": per_class,
           "mean_local_agreement": float(agree.mean()),
           "overlap_fraction": overlap_frac, "generalization_gap": gap,
           "error_by_margin_band": by_band.round(4).to_dict()}
    json.dump(res, open(out / "tables" / "bottleneck.json", "w"), indent=2, default=str)

    fig_bottleneck(per_class, agree, err, by_band, frozen, Atr, Ate, ytr, yte, out)

    banner("STAGE 9 | QUANTUM HYPOTHESIS")
    err_in = err[agree < 0.5].mean(); err_out = err[agree >= 0.5].mean()
    ratio = err_in / max(err_out, 1e-9)
    print(f"Observed: {100*(agree < 0.5).mean():.1f}% of test points lie in "
          f"neighbourhoods where\nthe majority of training neighbours carry the "
          f"opposite label. Error rate inside\nthose regions is {err_in:.3f} "
          f"against {err_out:.3f} outside, a factor of {ratio:.1f}.")
    print(f"\nModel capacity is not the binding constraint: the train-test gap "
          f"reaches\n{gap:+.3f} for the reference model, and higher-capacity "
          f"learners fit the training\nset almost perfectly without converting "
          f"that into test accuracy.")
    print("\nHypothesis under test: the limitation is the similarity geometry "
          "induced by\nthe input representation. A quantum feature map defines a "
          "different inner\nproduct over the same inputs, so the question is "
          "whether that geometry\nseparates the overlapping region any better "
          "than a classical kernel does.")
    print("\nThe matched-representation control in the quantum stage is what "
          "makes this\ntestable: a classical RBF-SVM fitted on the identical "
          "encoded features\nisolates the kernel's contribution from the "
          "encoding's.")
    return res


# ═══════════════════════════════════════════════════════════════════════════ #
# STAGE 10-14: QUANTUM PIPELINE
# ═══════════════════════════════════════════════════════════════════════════ #
def select_n_qubits(Atr, var_threshold=0.95, max_qubits=12):
    """Automatic component/qubit selection from the explained-variance curve,
    capped by the simulable qubit budget."""
    p = PCA().fit(Atr)
    cum = np.cumsum(p.explained_variance_ratio_)
    n_needed = int(np.searchsorted(cum, var_threshold) + 1)
    n_q = int(min(n_needed, Atr.shape[1], max_qubits))
    return n_q, n_needed, cum, p


def quantum_normalize(Ptr, others):
    """Map to [0, pi]: the ZZFeatureMap phase encoding is 2*pi periodic, so
    inputs outside this range alias onto each other."""
    mm = MinMaxScaler((0, np.pi)).fit(Ptr)
    f = lambda M: np.clip(mm.transform(M), 0, np.pi)
    return f(Ptr), [f(o) for o in others], mm


def quantum_pipeline(Atr, Ava, Ate, ytr, yva, yte, seed, out, max_train=3000,
                     quick=False, make_figs=True):
    banner("STAGE 10-14 | QUANTUM PIPELINE")
    rng = np.random.default_rng(seed)
    if len(Atr) > max_train:
        idx = rng.choice(len(Atr), max_train, replace=False)
        Atr_q, ytr_q = Atr[idx], ytr[idx]
        print(f"Training subsample for O(N^2) kernel: {len(Atr)} -> {max_train}")
    else:
        Atr_q, ytr_q = Atr, ytr

    # Pre-flight: confirm the quantum stage never sees validation or test data
    # while fitting. PCA and the [0, pi] scaler are fitted on Atr_q alone; the
    # validation partition scores configurations; the test partition is
    # transformed by already-fitted objects and touched once, after freezing.
    assert len(Atr_q) <= len(Atr), "quantum training subsample exceeds train set"
    print("  fit order verified: PCA(train) -> MinMaxScaler(train) -> "
          "kernel -> QSVM(train), scored on validation")

    n_q, n_needed, cum, pca_full = select_n_qubits(Atr_q)
    print(f"\nPCA representation:")
    print(f"  components for 95% variance: {n_needed}/{Atr.shape[1]}")
    print(f"  cumulative variance at 2/4/6/{Atr.shape[1]} comps: "
          + "/".join(f"{cum[min(k, len(cum))-1]:.3f}" for k in (2, 4, 6, Atr.shape[1])))
    print(f"  automatic qubit selection -> {n_q} qubits")

    mi_comp, _ = mi_of_components(Atr_q, ytr_q, seed=seed, out=out if make_figs else None)
    frac = mi_comp["MI_bits"][:n_q].sum() / max(mi_comp["MI_bits"].sum(), 1e-12)
    print(f"  MI retained by {n_q} components: {frac:.3f} of total")
    print(f"  MI is spread across components (max single-component share "
          f"{mi_comp['MI_bits'].max()/max(mi_comp['MI_bits'].sum(),1e-12):.3f}),")
    print("  so truncating the PCA discards target information directly.")

    if make_figs:
        fig_pca(pca_full, cum, Atr_q, ytr_q, n_q, out, mi_comp=mi_comp)
        fig_quantum_circuit(min(n_q, 5), reps=2, ent="linear", out=out)

    reps_grid = (2,) if quick else (1, 2)
    ent_grid = ("linear",) if quick else ("linear", "full")
    alpha_grid = (1.0, 2.0) if quick else (0.5, 1.0, 2.0)
    C_grid = (1, 10, 100)
    qubit_grid = sorted({max(2, n_q // 2), max(2, int(n_q * 0.75)), n_q})

    records, best = [], None
    for nq in qubit_grid:
        # Test data is deliberately absent from the search loop.
        if nq >= Atr_q.shape[1]:
            Ptr, Pothers = Atr_q, [Ava]
        else:
            pca = PCA(n_components=nq, random_state=seed).fit(Atr_q)
            Ptr, Pothers = pca.transform(Atr_q), [pca.transform(Ava)]
        Qtr, (Qva,), _ = quantum_normalize(Ptr, Pothers)

        # Matched-representation control: classical RBF on the identical encoded
        # features. Isolates kernel effect from encoding effect.
        control = max(balanced_accuracy_score(yva, SVC(C=C, gamma=g).fit(Qtr, ytr_q).predict(Qva))
                      for C in (1, 10, 100) for g in ("scale", 0.5))

        for reps in reps_grid:
            for ent in ent_grid:
                for a in alpha_grid:
                    t0 = time.time()
                    Ktr = fidelity_kernel(Qtr, reps=reps, ent=ent, alpha=a)
                    Kva = fidelity_kernel(Qva, Qtr, reps=reps, ent=ent, alpha=a)
                    kt = time.time() - t0
                    d = kernel_diagnostics(Ktr, ytr_q); kta = kernel_target_alignment(Ktr, ytr_q)
                    for C in C_grid:
                        svm = SVC(C=C, kernel="precomputed").fit(Ktr, ytr_q)
                        b = balanced_accuracy_score(yva, svm.predict(Kva))
                        rec = {"seed": seed, "n_qubits": nq, "reps": reps,
                               "entanglement": ent, "alpha": a, "C": C,
                               "val_bacc": b, "alignment": kta,
                               "classical_control": control,
                               "kernel_seconds": round(kt, 2), **d}
                        records.append(rec)
                        if best is None or b > best["val_bacc"]:
                            best = dict(rec)
        sel = [r for r in records if r["n_qubits"] == nq]
        print(f"  q={nq}: best_val={max(r['val_bacc'] for r in sel):.4f}  "
              f"(matched classical control {control:.4f})", flush=True)

    # Freeze, then touch the test set once.
    nq = best["n_qubits"]
    if nq >= Atr_q.shape[1]:
        Ptr, Pothers = Atr_q, [Ate]
    else:
        pca = PCA(n_components=nq, random_state=seed).fit(Atr_q)
        Ptr, Pothers = pca.transform(Atr_q), [pca.transform(Ate)]
    Qtr, (Qte,), _ = quantum_normalize(Ptr, Pothers)
    Ktr = fidelity_kernel(Qtr, reps=best["reps"], ent=best["entanglement"], alpha=best["alpha"])
    Kte = fidelity_kernel(Qte, Qtr, reps=best["reps"], ent=best["entanglement"], alpha=best["alpha"])
    svm = SVC(C=best["C"], kernel="precomputed").fit(Ktr, ytr_q)
    test_m = evaluate(yte, svm.predict(Kte), svm.decision_function(Kte))

    print(f"\nFrozen QSVM: qubits={nq} reps={best['reps']} ent={best['entanglement']} "
          f"alpha={best['alpha']} C={best['C']}")
    print(f"Test (untouched): acc={test_m['accuracy']:.4f} auc={test_m['roc_auc']:.4f} "
          f"f1={test_m['f1']:.4f} mcc={test_m['mcc']:.4f}")

    scores = svm.decision_function(Kte)
    return best, test_m, pd.DataFrame(records), scores


# ═══════════════════════════════════════════════════════════════════════════ #
# FIGURES (journal style)
# ═══════════════════════════════════════════════════════════════════════════ #
# Single-column 3.35 in, double-column 7.0 in (standard Elsevier/IEEE widths).
# Every figure is written as both PNG (600 dpi) and PDF (vector) so it can be
# dropped into a manuscript without resampling.
SINGLE, DOUBLE = 3.35, 7.0
PAL = {"c0": "#4C72B0", "c1": "#DD8452", "q": "#55A868",
       "acc": "#C44E52", "grey": "#8C8C8C"}


def journal_style():
    plt.rcParams.update({
        "figure.dpi": 120, "savefig.dpi": 600, "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8,
        "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
        "axes.linewidth": 0.6, "axes.edgecolor": "#333333",
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.4,
        "grid.color": "#BBBBBB",
        "xtick.major.width": 0.6, "ytick.major.width": 0.6,
        "xtick.major.size": 2.5, "ytick.major.size": 2.5,
        "lines.linewidth": 1.2, "lines.markersize": 3.5,
        "legend.frameon": False, "legend.handlelength": 1.5,
        "axes.titlepad": 4, "axes.labelpad": 2,
        "figure.constrained_layout.use": True,
        "figure.constrained_layout.h_pad": 0.04,
        "figure.constrained_layout.w_pad": 0.04,
        "mathtext.fontset": "dejavusans",
    })


SHORT = {"LogisticRegression": "LogReg", "RBF-SVM": "SVM",
         "RandomForest": "RF", "XGBoost": "XGB", "QSVM": "QSVM",
         "ExtraTrees": "ET", "KNN": "KNN", "MLP": "MLP"}


def short(n):
    return SHORT.get(n, n[:6])


def panel(ax, letter, dx=-0.16, dy=1.06):
    ax.text(dx, dy, f"({letter})", transform=ax.transAxes, fontsize=9,
            fontweight="bold", va="top", ha="left")


def save(fig, out, name):
    for ext in ("png", "pdf"):
        fig.savefig(out / "figures" / f"{name}.{ext}")
    plt.close(fig)


# --------------------------------------------------------------------------- #
# Mutual information
# --------------------------------------------------------------------------- #
def mutual_information(df, feats, seed=42, n_perm=200, out=None):
    """Mutual information between each feature and the target, in bits.

    MI captures any statistical dependence, not just monotone association, so it
    detects structure that a Spearman coefficient misses. Significance is
    established against a permutation null rather than an asymptotic
    approximation, because the k-NN MI estimator is biased upward at finite n
    and the bias does not vanish under the null.
    """
    from sklearn.feature_selection import mutual_info_classif

    X = df[feats].to_numpy(float)
    y = df[TARGET].to_numpy(int)
    NATS_TO_BITS = 1.0 / np.log(2)

    mi = mutual_info_classif(X, y, random_state=seed) * NATS_TO_BITS

    rng = np.random.default_rng(seed)
    null = np.empty((n_perm, len(feats)))
    for b in range(n_perm):
        null[b] = mutual_info_classif(X, rng.permutation(y),
                                      random_state=seed + b) * NATS_TO_BITS
    p = ((null >= mi).sum(axis=0) + 1) / (n_perm + 1)
    mi_corrected = np.clip(mi - null.mean(axis=0), 0, None)

    # Feature-feature MI, for redundancy: a pair with high MI carries
    # overlapping information even when their linear correlation is near zero.
    from sklearn.feature_selection import mutual_info_regression

    n = len(feats)
    mi_ff = np.zeros((n, n))
    for i in range(n):
        mi_ff[i] = mutual_info_regression(X, X[:, i], random_state=seed) * NATS_TO_BITS
    np.fill_diagonal(mi_ff, 0.0)
    mi_ff = 0.5 * (mi_ff + mi_ff.T)

    sp = df[feats + [TARGET]].corr(method="spearman")[TARGET][feats].abs().to_numpy()
    tab = pd.DataFrame({
        "feature": feats, "MI_bits": mi, "MI_null_mean": null.mean(axis=0),
        "MI_corrected_bits": mi_corrected, "p_permutation": p,
        "abs_spearman": sp,
        "MI_rank": pd.Series(-mi).rank().astype(int),
        "spearman_rank": pd.Series(-sp).rank().astype(int),
    }).sort_values("MI_bits", ascending=False)

    if out is not None:
        tab.to_csv(out / "tables" / "mutual_information.csv", index=False)
        pd.DataFrame(mi_ff, index=feats, columns=feats).to_csv(
            out / "tables" / "mutual_information_pairwise.csv")

    print("\nMutual information with target (bits, permutation-tested):")
    print(tab[["feature", "MI_bits", "MI_corrected_bits", "p_permutation",
               "abs_spearman"]].round(4).to_string(index=False))
    print(f"  total MI across features: {mi.sum():.4f} bits "
          f"(target entropy {_entropy(y):.4f} bits)")
    disagree = int((tab["MI_rank"] != tab["spearman_rank"]).sum())
    print(f"  {disagree}/{len(feats)} features rank differently under MI than "
          f"under |Spearman|,")
    print("  indicating dependence that is not monotone.")
    return tab, mi_ff


def _entropy(y):
    p = np.bincount(y) / len(y)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def mi_of_components(Atr, ytr, seed=42, out=None):
    """MI between each principal component and the target.

    This is the quantitative basis for the qubit budget: if MI is spread evenly
    across components, truncating the PCA discards target information directly.
    """
    from sklearn.feature_selection import mutual_info_classif

    p = PCA().fit(Atr)
    P = p.transform(Atr)
    mi = mutual_info_classif(P, ytr, random_state=seed) / np.log(2)
    tab = pd.DataFrame({
        "component": np.arange(1, P.shape[1] + 1),
        "explained_variance_ratio": p.explained_variance_ratio_,
        "MI_bits": mi,
        "MI_cumulative_frac": np.cumsum(mi) / max(mi.sum(), 1e-12),
    })
    if out is not None:
        tab.to_csv(out / "tables" / "mi_per_component.csv", index=False)
    return tab, p


# --------------------------------------------------------------------------- #
# Figure 1 | Dataset overview
# --------------------------------------------------------------------------- #
def fig_dataset_overview(df, feats, desc, out):
    fig, axes = plt.subplots(1, 3, figsize=(DOUBLE, 2.1))

    ax = axes[0]
    vc = df[TARGET].value_counts().sort_index()
    b = ax.bar(["Not potable", "Potable"], vc.values,
               color=[PAL["c0"], PAL["c1"]], width=0.6)
    for r, v in zip(b, vc.values):
        ax.text(r.get_x() + r.get_width() / 2, v, f"{v:,}\n{v/len(df):.1%}",
                ha="center", va="bottom", fontsize=6.5)
    ax.set_ylim(0, vc.max() * 1.28)
    ax.set_ylabel("Samples")
    panel(ax, "a")

    ax = axes[1]
    d = desc.sort_values("pct_outliers")
    ax.barh(d["feature"], d["pct_outliers"], color=PAL["c0"], height=0.65)
    ax.set_xlabel("Flagged outliers (%)")
    ax.tick_params(axis="y", labelsize=6)
    panel(ax, "b", dx=-0.55)

    ax = axes[2]
    ax.scatter(desc["skew"], desc["kurtosis"], s=22, color=PAL["q"],
               edgecolor="white", linewidth=0.4, zorder=3)
    for _, r in desc.iterrows():
        ax.annotate(r["feature"][:8], (r["skew"], r["kurtosis"]), fontsize=5.5,
                    xytext=(2.5, 2.5), textcoords="offset points", color="#444")
    ax.axhline(0, color=PAL["grey"], lw=0.5, ls="--")
    ax.axvline(0, color=PAL["grey"], lw=0.5, ls="--")
    ax.set_xlabel("Skewness")
    ax.set_ylabel("Excess kurtosis")
    panel(ax, "c")

    save(fig, out, "fig01_dataset_overview")


# --------------------------------------------------------------------------- #
# Figure 2 | Distributions
# --------------------------------------------------------------------------- #
def fig_distributions(df, feats, out):
    n = len(feats)
    ncol = 3
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(DOUBLE, 1.55 * nrow))
    for k, (ax, c) in enumerate(zip(np.ravel(axes), feats)):
        for cls, lab, col in [(0, "Not potable", PAL["c0"]),
                              (1, "Potable", PAL["c1"])]:
            sns.kdeplot(df.loc[df[TARGET] == cls, c].dropna(), ax=ax, fill=True,
                        alpha=0.28, color=col, lw=1.0, label=lab)
        ax.set_title(c, fontsize=7.5)
        ax.set_ylabel("Density" if k % ncol == 0 else "")
        ax.set_xlabel("")
        ax.tick_params(labelsize=6)
        if ax.get_legend():
            ax.get_legend().remove()
    for ax in np.ravel(axes)[n:]:
        ax.axis("off")
    h, l = np.ravel(axes)[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.03))
    save(fig, out, "fig02_distributions")


# --------------------------------------------------------------------------- #
# Figure 3 | Outlier analysis
# --------------------------------------------------------------------------- #
def fig_outliers(df, feats, desc, out):
    fig, ax = plt.subplots(figsize=(DOUBLE, 2.6))
    Z = (df[feats] - df[feats].mean()) / df[feats].std()
    Z[TARGET] = df[TARGET].map({0: "Not potable", 1: "Potable"})
    long = Z.melt(id_vars=TARGET, var_name="feature", value_name="z")
    sns.boxplot(data=long, x="feature", y="z", hue=TARGET, ax=ax,
                palette=[PAL["c0"], PAL["c1"]], fliersize=0.8, linewidth=0.6,
                width=0.7)
    ax.axhline(0, color=PAL["grey"], lw=0.5)
    ax.set_xticklabels([t.get_text() for t in ax.get_xticklabels()],
                       rotation=25, ha="right", fontsize=6.5)
    ax.set_xlabel("")
    ax.set_ylabel("Standardised value (z)")
    ax.legend(title="", loc="upper right", ncol=2)
    save(fig, out, "fig03_outliers")


# --------------------------------------------------------------------------- #
# Figure 4 | Correlation and mutual information
# --------------------------------------------------------------------------- #
def fig_correlation_mi(corr, mi_tab, mi_ff, feats, out):
    fig, axes = plt.subplots(1, 2, figsize=(DOUBLE, 3.0))

    ax = axes[0]
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
                vmin=-0.5, vmax=0.5, ax=ax, annot_kws={"size": 5},
                cbar_kws={"label": r"Spearman $\rho$", "shrink": 0.8,
                          "pad": 0.02}, linewidths=0.3, linecolor="white")
    ax.tick_params(labelsize=6)
    ax.set_title("Monotone association")
    panel(ax, "a", dx=-0.30)

    ax = axes[1]
    M = pd.DataFrame(mi_ff, index=feats, columns=feats)
    mask2 = np.triu(np.ones_like(M, dtype=bool), k=1)
    sns.heatmap(M, mask=mask2, annot=True, fmt=".2f", cmap="viridis", ax=ax,
                annot_kws={"size": 5}, cbar_kws={"label": "MI (bits)",
                                                 "shrink": 0.8, "pad": 0.02},
                linewidths=0.3, linecolor="white")
    ax.tick_params(labelsize=6)
    ax.set_title("Feature-feature mutual information")
    panel(ax, "b", dx=-0.30)

    save(fig, out, "fig04_correlation_mi")


# --------------------------------------------------------------------------- #
# Figure 5 | Mutual information with the target
# --------------------------------------------------------------------------- #
def fig_mutual_information(mi_tab, group, out):
    fig, axes = plt.subplots(1, 3, figsize=(DOUBLE, 2.3))

    ax = axes[0]
    t = mi_tab.sort_values("MI_bits")
    ax.barh(t["feature"], t["MI_bits"], color=PAL["q"], height=0.62,
            label="Observed MI")
    ax.barh(t["feature"], t["MI_null_mean"], color=PAL["grey"], height=0.62,
            alpha=0.85, label="Permutation null")
    ax.set_xlabel("Mutual information (bits)")
    ax.tick_params(axis="y", labelsize=6)
    ax.legend(loc="lower right")
    panel(ax, "a", dx=-0.55)

    ax = axes[1]
    ax.scatter(mi_tab["abs_spearman"], mi_tab["MI_bits"], s=22, color=PAL["c1"],
               edgecolor="white", linewidth=0.4, zorder=3)
    for _, r in mi_tab.iterrows():
        ax.annotate(r["feature"][:8], (r["abs_spearman"], r["MI_bits"]),
                    fontsize=5.5, xytext=(2.5, 2.5), textcoords="offset points",
                    color="#444")
    ax.set_xlabel(r"$|\rho_{\mathrm{Spearman}}|$ with target")
    ax.set_ylabel("MI with target (bits)")
    panel(ax, "b")

    ax = axes[2]
    g = group.set_index("feature")
    m = mi_tab.set_index("feature")
    common = [f for f in m.index if f in g.index]
    ax.scatter(g.loc[common, "cohens_d"].abs(), m.loc[common, "MI_bits"],
               s=22, color=PAL["c0"], edgecolor="white", linewidth=0.4, zorder=3)
    for f in common:
        ax.annotate(f[:8], (abs(g.loc[f, "cohens_d"]), m.loc[f, "MI_bits"]),
                    fontsize=5.5, xytext=(2.5, 2.5), textcoords="offset points",
                    color="#444")
    ax.set_xlabel(r"$|d|$ (mean-shift effect size)")
    ax.set_ylabel("MI with target (bits)")
    panel(ax, "c")

    save(fig, out, "fig05_mutual_information")


# --------------------------------------------------------------------------- #
# Figure 6 | Bottleneck analysis
# --------------------------------------------------------------------------- #
def fig_bottleneck(per_class, agree, err, by_band, frozen, Atr, Ate, ytr, yte, out):
    fig, axes = plt.subplots(1, 4, figsize=(DOUBLE, 2.0))

    ax = axes[0]
    b = ax.bar(["0", "1"], [p["error_rate"] for p in per_class],
               color=[PAL["c0"], PAL["c1"]], width=0.55)
    for r, p in zip(b, per_class):
        ax.text(r.get_x() + r.get_width() / 2, p["error_rate"],
                f"{p['error_rate']:.3f}", ha="center", va="bottom", fontsize=6.5)
    ax.set_xlabel("Class")
    ax.set_ylabel("Error rate")
    ax.set_ylim(0, max(p["error_rate"] for p in per_class) * 1.3)
    panel(ax, "a", dx=-0.34)

    ax = axes[1]
    bins = np.linspace(0, 1, 21)
    ax.hist(agree[~err], bins=bins, color=PAL["q"], alpha=0.75, label="Correct")
    ax.hist(agree[err], bins=bins, color=PAL["acc"], alpha=0.75,
            label="Misclassified")
    ax.axvline(0.5, color="#333", ls="--", lw=0.7)
    ax.set_xlabel("Local label agreement")
    ax.set_ylabel("Test samples")
    ax.legend(loc="upper left")
    panel(ax, "b", dx=-0.34)

    ax = axes[2]
    lo, hi = err[agree < 0.5].mean(), err[agree >= 0.5].mean()
    b = ax.bar(["Overlap", "Clean"], [lo, hi], color=[PAL["acc"], PAL["q"]],
               width=0.55)
    for r, v in zip(b, [lo, hi]):
        ax.text(r.get_x() + r.get_width() / 2, v, f"{v:.3f}", ha="center",
                va="bottom", fontsize=6.5)
    ax.set_ylabel("Error rate")
    ax.set_ylim(0, max(lo, hi) * 1.3)
    ax.set_xlabel("Neighbourhood")
    panel(ax, "c", dx=-0.34)

    ax = axes[3]
    names = list(frozen)
    tr = [balanced_accuracy_score(ytr, frozen[n].predict(Atr)) for n in names]
    te = [balanced_accuracy_score(yte, frozen[n].predict(Ate)) for n in names]
    x = np.arange(len(names))
    w = 0.36
    ax.bar(x - w / 2, tr, w, label="Train", color=PAL["c0"])
    ax.bar(x + w / 2, te, w, label="Test", color=PAL["c1"])
    ax.set_xticks(x)
    ax.set_xticklabels([short(n) for n in names], rotation=30, ha="right",
                       fontsize=6.5)
    ax.set_ylim(0.5, 1.18)
    ax.set_yticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
    ax.set_ylabel("Balanced accuracy")
    ax.legend(loc="upper center", ncol=2, fontsize=6, columnspacing=0.9,
              handlelength=1.1, borderaxespad=0.1)
    panel(ax, "d", dx=-0.34)

    save(fig, out, "fig06_bottleneck")


# --------------------------------------------------------------------------- #
# Figure 7 | PCA representation and MI per component
# --------------------------------------------------------------------------- #
def fig_pca(pca_full, cum, Atr, ytr, n_q, out, mi_comp=None):
    fig, axes = plt.subplots(1, 4, figsize=(DOUBLE, 2.0))

    ax = axes[0]
    k = np.arange(1, len(cum) + 1)
    ax.bar(k, pca_full.explained_variance_ratio_, color=PAL["c0"], width=0.62)
    ax.plot(k, cum, "o-", color=PAL["acc"], ms=2.5, lw=1.0)
    ax.axhline(0.95, color=PAL["grey"], ls="--", lw=0.7)
    ax.axvline(n_q, color=PAL["q"], ls=":", lw=1.1)
    ax.set_xlabel("Component")
    ax.set_ylabel("Explained variance")
    ax.set_xticks(k[::2])
    panel(ax, "a", dx=-0.34)

    ax = axes[1]
    P = pca_full.transform(Atr)
    for c, lab, col in [(0, "Not potable", PAL["c0"]), (1, "Potable", PAL["c1"])]:
        s = P[ytr == c]
        ax.scatter(s[:, 0], s[:, 1], s=1.6, alpha=0.30, color=col, label=lab,
                   linewidths=0)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.legend(loc="upper right", markerscale=4, handletextpad=0.3)
    panel(ax, "b", dx=-0.34)

    ax = axes[2]
    sep = []
    for i in range(P.shape[1]):
        a, b_ = P[ytr == 0, i], P[ytr == 1, i]
        sep.append(abs(a.mean() - b_.mean()) / np.sqrt((a.var() + b_.var()) / 2))
    ax.bar(k, sep, color=PAL["c1"], width=0.62)
    ax.set_xlabel("Component")
    ax.set_ylabel(r"$|d|$ between classes")
    ax.set_xticks(k[::2])
    panel(ax, "c", dx=-0.34)

    ax = axes[3]
    if mi_comp is not None:
        ax.bar(mi_comp["component"], mi_comp["MI_bits"], color=PAL["q"],
               width=0.62)
        ax.axvline(n_q, color=PAL["acc"], ls=":", lw=1.1)
        ax.set_xlabel("Component")
        ax.set_ylabel("MI with target (bits)")
        ax.set_xticks(mi_comp["component"][::2])
    else:
        Q = MinMaxScaler((0, np.pi)).fit_transform(P[:, :n_q])
        ax.hist(Q.ravel(), bins=50, color=PAL["q"])
        ax.set_xticks([0, np.pi / 2, np.pi])
        ax.set_xticklabels(["0", r"$\pi/2$", r"$\pi$"])
        ax.set_xlabel("Encoded value")
        ax.set_ylabel("Count")
    panel(ax, "d", dx=-0.34)

    save(fig, out, "fig07_pca_representation")


# --------------------------------------------------------------------------- #
# Figure 8 | Quantum circuit
# --------------------------------------------------------------------------- #
def fig_quantum_circuit(n_qubits, reps, ent, out):
    """ZZFeatureMap circuit, drawn directly so no Qiskit dependency is needed."""
    pairs = entangling_pairs(n_qubits, ent)
    fig, ax = plt.subplots(figsize=(DOUBLE, 0.42 * n_qubits + 0.75))
    fig.set_constrained_layout(False)

    def box(x, q, label, color, w=0.62, h=0.34, fs=5.5):
        ax.add_patch(mpatches.FancyBboxPatch(
            (x - w / 2, q - h / 2), w, h, boxstyle="round,pad=0.015",
            fc=color, ec="#222", lw=0.7, zorder=3))
        ax.text(x, q, label, ha="center", va="center", fontsize=fs, zorder=4)

    def cnot(x, i, j):
        ax.plot([x, x], [i, j], color="#222", lw=0.8, zorder=2)
        ax.scatter([x], [i], s=11, color="#222", zorder=4)
        ax.add_patch(mpatches.Circle((x, j), 0.115, fc="white", ec="#222",
                                     lw=0.8, zorder=4))
        ax.plot([x - .115, x + .115], [j, j], color="#222", lw=0.7, zorder=5)
        ax.plot([x, x], [j - .115, j + .115], color="#222", lw=0.7, zorder=5)

    x = 0.7
    for r in range(reps):
        for q in range(n_qubits):
            box(x, q, "H", "#F2E2C4")
        x += 0.95
        for q in range(n_qubits):
            box(x, q, r"$P(2x_{%d})$" % q, "#CBDCEC", w=0.88)
        x += 1.1
        for (i, j) in pairs:
            cnot(x, i, j); x += 0.72
            box(x, j, r"$P(2\phi_{%d%d})$" % (i, j), "#CFE6D8", w=0.95)
            x += 0.72
            cnot(x, i, j); x += 0.85
        if r < reps - 1:
            ax.axvline(x - 0.42, color="#BBB", ls=":", lw=0.7)

    for q in range(n_qubits):
        ax.plot([0.1, x - 0.2], [q, q], color="#333", lw=0.7, zorder=1)
        ax.text(-0.05, q, rf"$q_{q}$", ha="right", va="center", fontsize=7)

    ax.set_xlim(-0.45, x - 0.1)
    ax.set_ylim(-0.55, n_qubits - 0.45)
    ax.invert_yaxis()
    ax.axis("off")
    ax.set_title(rf"$n={n_qubits}$ qubits, {reps} repetition(s), {ent} "
                 rf"entanglement; $\phi_{{ij}}=(\pi-x_i)(\pi-x_j)$; "
                 rf"{2*len(pairs)*reps} two-qubit gates", fontsize=7, pad=6)
    fig.tight_layout()
    save(fig, out, "fig08_quantum_circuit")


# --------------------------------------------------------------------------- #
# Figure 9 | ROC and precision-recall
# --------------------------------------------------------------------------- #
def fig_roc_pr(preds, out):
    fig, axes = plt.subplots(1, 2, figsize=(DOUBLE, 2.7))
    cols = {n: (PAL["q"] if n == "QSVM" else c) for n, c in
            zip(preds, [PAL["c0"], PAL["c1"], PAL["acc"], "#937860", PAL["grey"]])}

    ax = axes[0]
    for name, (yt, ys) in preds.items():
        fpr, tpr, _ = roc_curve(yt, ys)
        ax.plot(fpr, tpr, ls="--" if name == "QSVM" else "-",
                lw=1.4 if name == "QSVM" else 1.0, color=cols[name],
                label=f"{name} ({roc_auc_score(yt, ys):.3f})")
    ax.plot([0, 1], [0, 1], ":", color=PAL["grey"], lw=0.7)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.legend(loc="lower right", title="AUC", title_fontsize=7)
    panel(ax, "a")

    ax = axes[1]
    for name, (yt, ys) in preds.items():
        pr, rc, _ = precision_recall_curve(yt, ys)
        ax.plot(rc, pr, ls="--" if name == "QSVM" else "-",
                lw=1.4 if name == "QSVM" else 1.0, color=cols[name],
                label=f"{name} ({average_precision_score(yt, ys):.3f})")
    base = float(np.mean(list(preds.values())[0][0]))
    ax.axhline(base, ls=":", color=PAL["grey"], lw=0.7)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(loc="lower left", title="AP", title_fontsize=7)
    panel(ax, "b")

    save(fig, out, "fig09_roc_pr")


# --------------------------------------------------------------------------- #
# Figure 10 | Confusion matrices
# --------------------------------------------------------------------------- #
def fig_confusion(preds, out):
    n = len(preds)
    fig, axes = plt.subplots(1, n, figsize=(DOUBLE, 1.75))
    for ax, (name, (yt, ys)) in zip(np.ravel(axes), preds.items()):
        thr = 0.5 if np.min(ys) >= 0 else 0.0
        pred = (np.asarray(ys) >= thr).astype(int)
        cm = confusion_matrix(yt, pred, labels=[0, 1])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                    xticklabels=["0", "1"], yticklabels=["0", "1"],
                    annot_kws={"size": 7}, linewidths=0.4, linecolor="white",
                    square=True)
        ax.set_title(f"{short(name)}\nacc = {accuracy_score(yt, pred):.3f}",
                     fontsize=7)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True" if ax is np.ravel(axes)[0] else "")
        ax.tick_params(labelsize=6)
    save(fig, out, "fig10_confusion")


# --------------------------------------------------------------------------- #
# Figure 11 | Classical vs QSVM, robustness and ablations
# --------------------------------------------------------------------------- #
def fig_comparison(summary, res, search, out):
    fig, axes = plt.subplots(2, 2, figsize=(DOUBLE, 4.4))

    ax = axes[0, 0]
    models = list(summary.index)
    x = np.arange(len(METRICS))
    w = 0.8 / len(models)
    base = [PAL["c0"], PAL["c1"], PAL["acc"], "#937860", PAL["grey"]]
    for i, m in enumerate(models):
        col = PAL["q"] if m == "QSVM" else base[i % len(base)]
        ax.bar(x + i * w - 0.4 + w / 2, [summary.loc[m, k] for k in METRICS], w,
               label=m, color=col, edgecolor="white", linewidth=0.3)
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace("_", " ").replace("balanced", "bal.")
                        for m in METRICS], rotation=35, ha="right", fontsize=6)
    ax.set_ylim(0.5, 1.0)
    ax.set_ylabel("Score")
    ax.legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.02),
              fontsize=6, columnspacing=1.0, handlelength=1.1)
    panel(ax, "a", dy=1.30)

    ax = axes[0, 1]
    acc = summary["accuracy"].sort_values()
    cols = [PAL["q"] if m == "QSVM" else PAL["c0"] for m in acc.index]
    ax.barh([short(m) for m in acc.index], acc.values,
            xerr=summary.loc[acc.index, "accuracy_sd"],
            color=cols, height=0.6, error_kw={"lw": 0.8, "capsize": 2.5,
                                              "ecolor": "#222"})
    ax.set_xlim(0.78, min(1.0, acc.max() + 0.03))
    ax.set_xlabel("Test accuracy")
    ax.tick_params(axis="y", labelsize=6.5)
    panel(ax, "b", dx=-0.42)

    ax = axes[1, 0]
    g = search.groupby("n_qubits").agg(q=("val_bacc", "max"),
                                       c=("classical_control", "max"))
    ax.plot(g.index, g["q"], "o-", color=PAL["q"], label="Quantum kernel")
    ax.plot(g.index, g["c"], "s--", color=PAL["c0"],
            label="Classical RBF, same features")
    ax.set_xlabel("Qubits (encoded dimensions)")
    ax.set_ylabel("Validation bal. accuracy")
    ax.set_xticks(g.index)
    ax.legend(loc="lower right")
    panel(ax, "c")

    ax = axes[1, 1]
    b = search.groupby("alpha").agg(v=("val_bacc", "max"),
                                    r=("effective_rank", "mean"))
    ax.plot(b.index, b["v"], "o-", color=PAL["q"])
    ax.set_xlabel(r"Bandwidth $\alpha$")
    ax.set_ylabel("Validation bal. accuracy", color=PAL["q"])
    ax.tick_params(axis="y", colors=PAL["q"])
    ax2 = ax.twinx()
    ax2.plot(b.index, b["r"], "s--", color=PAL["c0"])
    ax2.set_ylabel("Kernel effective rank", color=PAL["c0"])
    ax2.tick_params(axis="y", colors=PAL["c0"])
    ax2.grid(False)
    ax2.spines["right"].set_visible(True)
    panel(ax, "d")

    save(fig, out, "fig11_classical_vs_qsvm")


# ═══════════════════════════════════════════════════════════════════════════ #
# MAIN
# ═══════════════════════════════════════════════════════════════════════════ #
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default=None)
    ap.add_argument("--seeds", nargs="+", type=int, default=[42, 7, 2024])
    ap.add_argument("--max-train", type=int, default=3000)
    ap.add_argument("--outdir", default="results")
    ap.add_argument("--quick", action="store_true")
    args = ap.parse_args(_cli_args())

    out = Path(args.outdir)
    (out / "figures").mkdir(parents=True, exist_ok=True)
    (out / "tables").mkdir(parents=True, exist_ok=True)
    t0 = time.time()

    df = load_dataset(args.data)
    feats = [c for c in df.columns if c != TARGET]
    journal_style()
    df = quality_control(df, feats, out)

    X, y = df[feats].to_numpy(float), df[TARGET].to_numpy(int)

    all_rows, searches, first = [], [], True
    for seed in args.seeds:
        banner(f"SEED {seed}")
        Xtr, Xva, Xte, ytr, yva, yte = stratified_split(X, y, seed)
        print(f"Stratified split: train={len(ytr)} val={len(yva)} test={len(yte)}")

        Atr, Ava, Ate, _ = fit_preprocessing(Xtr, Xva, Xte)
        print("Preprocessing: median imputer + StandardScaler fitted on train only")
        if first:
            leakage_audit(len(ytr), len(yva), len(yte))

            # EDA on the training partition. Running it on the full table would
            # let test-set structure inform reported findings even when no model
            # consumes them.
            eda_df = pd.DataFrame(Xtr, columns=feats)
            eda_df[TARGET] = ytr
            eda_df = eda_df.fillna(eda_df.median(numeric_only=True))
            run_eda(eda_df, feats, out, partition_label=f"training partition, seed {seed}")

        banner("IMBALANCE STRATEGY SELECTION (validation)")
        imb, imb_tab = select_imbalance_strategy(Atr, Ava, ytr, yva, seed)
        if first:
            imb_tab.to_csv(out / "tables" / "imbalance_selection.csv", index=False)

        cres, frozen, val_scores = classical_pipeline(
            Atr, Ava, Ate, ytr, yva, yte, seed, args.quick, imbalance=imb)
        cres["seed"] = seed
        cres["imbalance"] = imb
        all_rows.append(cres)

        if first:
            bottleneck_analysis(frozen, val_scores, Atr, Ate, ytr, yte, feats, out)

        best, qm, search, qscores = quantum_pipeline(
            Atr, Ava, Ate, ytr, yva, yte, seed, out,
            max_train=args.max_train, quick=args.quick, make_figs=first)
        all_rows.append(pd.DataFrame([{
            "model": "QSVM", "seed": seed, "val_bacc": best["val_bacc"],
            "imbalance": imb,
            "params": json.dumps({k: best[k] for k in
                                  ("n_qubits", "reps", "entanglement", "alpha", "C")}),
            **qm}]))
        search["seed"] = seed
        searches.append(search)

        if first:
            preds = {n: (yte, _score(m, Ate)) for n, m in frozen.items()}
            preds["QSVM"] = (yte, qscores)
            fig_roc_pr(preds, out)
            fig_confusion(preds, out)
            first = False

    res = pd.concat(all_rows, ignore_index=True)
    res.to_csv(out / "tables" / "raw_results.csv", index=False)
    search = pd.concat(searches, ignore_index=True)
    search.to_csv(out / "tables" / "quantum_search.csv", index=False)

    banner("DIRECT COMPARISON | CLASSICAL vs QSVM (untouched test set)")
    g = res.groupby("model")
    summary = g[METRICS].mean()
    sds = g[METRICS].std().fillna(0.0)
    for m in METRICS:
        summary[f"{m}_sd"] = sds[m]
    summary = summary.sort_values("accuracy", ascending=False)
    summary.to_csv(out / "tables" / "comparison_summary.csv")

    pretty = pd.DataFrame(index=summary.index)
    for m in METRICS:
        pretty[m] = (summary[m].round(4).astype(str) + " ± "
                     + summary[f"{m}_sd"].round(4).astype(str))
    pretty.to_csv(out / "tables" / "comparison_formatted.csv")
    print(pretty.to_string())

    banner("STATISTICAL ANALYSIS")
    wide = res.pivot_table(index="seed", columns="model", values="accuracy")
    if wide.shape[1] >= 3 and wide.shape[0] >= 3:
        stat, p = stats.friedmanchisquare(*[wide[c].to_numpy() for c in wide.columns])
        ranks = wide.rank(axis=1, ascending=False).mean().sort_values()
        print(f"Friedman chi2={stat:.3f}  p={p:.4g}")
        print("Mean ranks (lower is better):")
        print(ranks.round(3).to_string())
        ranks.to_csv(out / "tables" / "mean_ranks.csv")
    else:
        print(f"Friedman needs >=3 models and >=3 seeds; have "
              f"{wide.shape[1]} models, {wide.shape[0]} seeds")

    if "QSVM" in wide.columns and wide.shape[1] > 1:
        others = [c for c in wide.columns if c != "QSVM"]
        top = wide[others].mean().idxmax()
        d = wide["QSVM"] - wide[top]
        n = len(d)
        if n > 1 and d.std(ddof=1) > 0:
            n_te, n_tr = int(.2 * len(y)), int(.6 * len(y))
            t = d.mean() / np.sqrt(((1/n) + (n_te/n_tr)) * d.var(ddof=1))
            pv = 2 * (1 - stats.t.cdf(abs(t), n - 1))
            print(f"\nQSVM vs {top} (Nadeau-Bengio corrected t-test):")
            print(f"  mean difference {d.mean():+.4f}  t={t:.3f}  p={pv:.4f}")
        else:
            print(f"\nQSVM vs {top}: mean difference {d.mean():+.4f} "
                  f"(need >=2 seeds for a test)")

    banner("MULTI-SEED ROBUSTNESS")
    print(res.pivot_table(index="seed", columns="model",
                          values="accuracy").round(4).to_string())

    banner("ABLATION STUDIES")
    print("Encoding (qubit count) vs matched classical control:")
    print(search.groupby("n_qubits").agg(
        quantum=("val_bacc", "max"),
        classical_control=("classical_control", "max")).round(4).to_string())
    print("\nBandwidth alpha vs kernel geometry:")
    print(search.groupby("alpha").agg(
        val_bacc=("val_bacc", "max"), offdiag_mean=("offdiag_mean", "mean"),
        effective_rank=("effective_rank", "mean"),
        alignment=("alignment", "mean")).round(4).to_string())
    print("\nEntanglement topology:")
    print(search.groupby("entanglement")["val_bacc"].max().round(4).to_string())
    print("\nCircuit repetitions:")
    print(search.groupby("reps")["val_bacc"].max().round(4).to_string())

    fig_comparison(summary, res, search, out)

    json.dump({"seeds": args.seeds, "n_samples": int(len(X)),
               "n_features": len(feats),
               "best_model": summary.index[0],
               "best_accuracy": float(summary.iloc[0]["accuracy"]),
               "qsvm_accuracy": float(summary.loc["QSVM", "accuracy"])
               if "QSVM" in summary.index else None,
               "runtime_minutes": round((time.time() - t0) / 60, 2)},
              open(out / "manifest.json", "w"), indent=2)

    print(f"\nDone in {(time.time()-t0)/60:.1f} min. Artefacts -> {out.resolve()}")
    print("Figures (PNG at 600 dpi + vector PDF):")
    for f in ["fig01 dataset overview", "fig02 distributions", "fig03 outliers",
              "fig04 correlation + pairwise MI", "fig05 mutual information",
              "fig06 bottleneck", "fig07 PCA representation + MI per component",
              "fig08 quantum circuit", "fig09 ROC / precision-recall",
              "fig10 confusion matrices", "fig11 classical vs QSVM + ablations"]:
        print(f"  {f}")


if __name__ == "__main__":
    main()



STAGE 1 | DATASET
Path to dataset files: /home/score/.cache/kagglehub/datasets/tamilamudan/water-quality-potability/versions/1
Loaded water_quality_potability.csv: 10000 rows x 10 columns

STAGE 3 | DATA QUALITY CONTROL (non-learned)
Duplicate removal: 10000 -> 10000 rows
Implausible values recoded: 0
Missing cells remaining: 0 (imputed later, fitted on training partition only)

SEED 42
Stratified split: train=6000 val=2000 test=2000
Preprocessing: median imputer + StandardScaler fitted on train only

Leakage audit — every fitted object and its fitting partition:
  SimpleImputer (median)               fitted on: train
  StandardScaler                       fitted on: train
  Mutual information                   fitted on: train
  PCA (component/qubit selection)      fitted on: train
  MinMaxScaler -> [0, pi]              fitted on: train
  Classical hyperparameters            fitted on: validation
  Imbalance strategy                   fitted on: validation
  Bottleneck reference mode